# 02 — Feature Engineering

Build and inspect:
- Champion encoder (ID → dense index)
- Draft-state feature vectors
- Synergy and counter matrices


In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.preprocess import load_processed
from src.features.champion_encoder import ChampionEncoder, DraftStateEncoder
from src.features.synergy_counter import build_synergy_matrix, build_counter_matrix

df = load_processed()
all_ids = sorted(df['champion_id'].unique().tolist())
champ_enc = ChampionEncoder(all_ids)
state_enc = DraftStateEncoder(champ_enc)
print(f'Vocabulary size: {champ_enc.num_champions}')
print(f'Feature vector dim: {state_enc.feature_dim}')

In [ ]:
# Inspect a single feature vector
sample = df.iloc[0]
vec = state_enc.encode(
    blue_picks=sample['blue_picks_so_far'],
    red_picks=sample['red_picks_so_far'],
    blue_bans=sample['blue_bans'],
    red_bans=sample['red_bans'],
    pick_order=sample['pick_order'],
    team=sample['team'],
)
print(f'Shape: {vec.shape},  Non-zero: {(vec != 0).sum()}')

In [ ]:
# Build synergy matrix (may take a few minutes on large datasets)
synergy = build_synergy_matrix(df, champ_enc)
counter = build_counter_matrix(df, champ_enc)
print('Synergy matrix shape:', synergy.shape)
print('Mean synergy win-rate (non-zero):', synergy[synergy > 0].mean())

In [ ]:
# Visualise top-50 champion synergy sub-matrix
N = 50
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

sns.heatmap(synergy[:N, :N], ax=axes[0], cmap='RdYlGn', center=0.5,
            xticklabels=False, yticklabels=False, vmin=0.3, vmax=0.7)
axes[0].set_title('Synergy Matrix (top-50 champions)')

sns.heatmap(counter[:N, :N], ax=axes[1], cmap='RdYlGn', center=0.5,
            xticklabels=False, yticklabels=False, vmin=0.3, vmax=0.7)
axes[1].set_title('Counter Matrix (top-50 champions)')

plt.tight_layout()
plt.show()